In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "train.txt").is_file():
    PROJECT_FOLDER = PROJECT_FOLDER / "NLP" / "Project"

DATA_PATH = PROJECT_FOLDER / "train.txt"
MODEL_PATH = PROJECT_FOLDER / "Emotion_Detector_Model.pkl"
df = pd.read_csv(DATA_PATH, sep=";", header=None, names=["text", "emotion"])
df.head()

In [ ]:
df['emotion'].unique()

In [ ]:
df["emotion"].value_counts()

In [ ]:
df.loc[1, "text"]

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"].astype(str),
    df["emotion"].astype(str),
    test_size=0.20,
    random_state=42,
    stratify=df["emotion"],
)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow,y_train)

pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test,pred_bow))


In [ ]:
pred_bow

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [ ]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

In [ ]:
y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test,y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
logictic_model = LogisticRegression(max_iter=1000)
logictic_model.fit(X_train_tfidf,y_train)

In [ ]:
lg_red =logictic_model.predict(X_test_tfidf)

In [ ]:
print(accuracy_score(y_test,lg_red))

In [ ]:
import joblib

emotion_model = Pipeline([
    ("tfidf", tfidf_vectorizer),
    ("classifier", logictic_model),
])

joblib.dump(emotion_model, MODEL_PATH)
print(f"Saved the trained pipeline to: {MODEL_PATH}")